# Business Intelligence and Analytics — Assignment Two
**Dataset:** BI_Retail_Case_Study.xlsx

This notebook works through the four case studies (Store Performance, Customer Value,
Product Profitability, Sales Channel & Operations) plus the final integration question.

Each question has a short text cell explaining **what I expect to find and why**, followed
by the code that investigates it. Business interpretation is written after looking at the
actual output, since it must be grounded in what the numbers actually show.

## Setup — Mount Drive

Colab runs on a temporary cloud machine, so Google Drive has to be mounted every session
before the dataset can be read from it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Setup — Libraries

- **pandas** — loading the Excel sheets, joining tables, grouping and aggregating.
- **numpy** — numerical operations (logs, conditional logic) used in a few of the deeper checks.
- **matplotlib** — base plotting engine, used for full control over chart layout.
- **seaborn** — nicer statistical charts (scatter with hue, box plots, heatmaps) built on matplotlib.
- **scipy.stats** — correlation tests (Spearman) used where the relationship isn't purely linear.

The display options just make wide tables and long decimals easier to read, they don't
affect any calculation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

# Make tables easier to read
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Use a clean plotting style
sns.set_theme(style="whitegrid")

## Setup — Load the File

All seven sheets are loaded once here, then reused throughout the notebook. Loading
everything up front avoids repeating `pd.read_excel` calls in every question.

In [ ]:
FILE = "/content/drive/MyDrive/BI/BI_Retail_Case_Study.xlsx"

sales     = pd.read_excel(FILE, sheet_name="SalesRaw")
products  = pd.read_excel(FILE, sheet_name="DimProduct")
customers = pd.read_excel(FILE, sheet_name="DimCustomer")
stores    = pd.read_excel(FILE, sheet_name="DimStore")
employees = pd.read_excel(FILE, sheet_name="DimEmployee")
dates     = pd.read_excel(FILE, sheet_name="DimDate")
targets   = pd.read_excel(FILE, sheet_name="Targets")

sales["TransactionDate"] = pd.to_datetime(sales["TransactionDate"]).dt.normalize()

print("SalesRaw:   ", sales.shape)
print("DimProduct: ", products.shape)
print("DimCustomer:", customers.shape)
print("DimStore:   ", stores.shape)
print("DimEmployee:", employees.shape)
print("DimDate:    ", dates.shape)
print("Targets:    ", targets.shape)

sales.head()

## Setup — Build One Master Transaction Table

Most questions need SalesRaw joined with product, store, employee and customer details, plus
Gross Profit (which is not stored directly, it has to be calculated as NetSales minus cost).
Building this once here means every question below can reuse `master` instead of repeating
the same joins.

In [ ]:
master = sales.merge(products[["ProductID", "ProductName", "Category", "UnitCost"]],
                      on="ProductID", how="left")
master = master.merge(stores[["StoreID", "StoreName", "StoreType", "Province"]],
                       on="StoreID", how="left")
master = master.merge(employees[["EmployeeID", "EmployeeName", "Role"]],
                       on="EmployeeID", how="left")
master = master.merge(customers[["CustomerID", "CustomerType", "Gender", "Age",
                                  "District", "RegistrationDate"]],
                       on="CustomerID", how="left")

master["TotalCost"]   = master["Quantity"] * master["UnitCost"]
master["GrossProfit"] = master["NetSales"] - master["TotalCost"]
master["YearMonth"]   = master["TransactionDate"].dt.strftime("%Y-%m")

# Sanity check: make sure every join found a match. These should all print 0.
print("Sales rows with no product match: ", master["ProductName"].isna().sum())
print("Sales rows with no store match:   ", master["StoreName"].isna().sum())
print("Sales rows with no employee match:", master["EmployeeName"].isna().sum())
print("Sales rows with no customer match:", master["CustomerType"].isna().sum())

master.head()

# Case Study One — Store Performance and Target Achievement (8 marks)

**The question management is really asking:** is the store with the most sales also the
store that is actually performing best once profit, efficiency and targets are considered?

## Question 1 (4 marks) — Store-Level KPI Table

**What I expect to find and why:** a store can rank #1 on Net Sales simply by being the
busiest (most transactions), while a smaller store could still be doing well on margin or
target achievement. So I expect the sales leader and the "strongest performer" to not
automatically be the same store once all the KPIs are looked at together, similar to what
Question 3 of Assignment One showed.

**What the code does:** aggregates the master table to one row per store with Net Sales,
Transactions, Quantity Sold, Gross Profit and Profit Margin, then brings in each store's
annual Sales Target from the Targets sheet (summed across its 12 months) to calculate
Target Achievement %.

In [ ]:
# Actual performance per store, for the full year
store_actual = master.groupby(["StoreID", "StoreName", "StoreType", "Province"]).agg(
    NetSales=("NetSales", "sum"),
    Transactions=("TransactionID", "nunique"),
    QuantitySold=("Quantity", "sum"),
    GrossProfit=("GrossProfit", "sum")
).reset_index()

store_actual["ProfitMargin"]         = store_actual["GrossProfit"] / store_actual["NetSales"]
store_actual["AvgTransactionValue"]  = store_actual["NetSales"] / store_actual["Transactions"]

# Convert Targets.Month into YearMonth so it can be matched and summed per store
m = targets["Month"]
if pd.api.types.is_numeric_dtype(m) and m.between(1, 12).all():
    year = master["TransactionDate"].dt.year.mode()[0]
    targets["YearMonth"] = f"{year}-" + m.astype(int).astype(str).str.zfill(2)
else:
    targets["YearMonth"] = pd.to_datetime(m.astype(str), errors="coerce").dt.strftime("%Y-%m")

store_annual_target = targets.groupby("StoreID")["SalesTarget"].sum().reset_index(name="SalesTarget")

# Combine actuals with targets
store_kpi = store_actual.merge(store_annual_target, on="StoreID", how="left")
store_kpi["AchievementPct"] = store_kpi["NetSales"] / store_kpi["SalesTarget"] * 100

# Rank each store on every KPI (1 = best)
for col, name in [("NetSales", "SalesRank"), ("GrossProfit", "ProfitRank"),
                  ("ProfitMargin", "MarginRank"), ("AvgTransactionValue", "ATVRank"),
                  ("AchievementPct", "AchievementRank")]:
    store_kpi[name] = store_kpi[col].rank(ascending=False, method="min").astype(int)

store_kpi = store_kpi.sort_values("NetSales", ascending=False).reset_index(drop=True)

cols = ["StoreName", "StoreType", "Province", "NetSales", "Transactions", "QuantitySold",
        "GrossProfit", "ProfitMargin", "AvgTransactionValue", "SalesTarget", "AchievementPct"]
print("STORE-LEVEL KPI TABLE")
display(store_kpi[cols].style.format({
    "NetSales": "{:,.0f}", "GrossProfit": "{:,.0f}", "SalesTarget": "{:,.0f}",
    "ProfitMargin": "{:.2%}", "AvgTransactionValue": "{:,.0f}", "AchievementPct": "{:.1f}"
}))

rank_cols = ["StoreName", "SalesRank", "ProfitRank", "MarginRank", "ATVRank", "AchievementRank"]
print("RANK OF EACH STORE ON EACH KPI (1 = best)")
display(store_kpi[rank_cols])

**How to interpret this:** find the store ranked #1 on `SalesRank` and check its rank on
the other four columns. If it stays near the top everywhere, the sales leader genuinely is
the strongest all-round performer, use at least three of the KPIs (for example NetSales,
ProfitMargin and AchievementPct) to justify that in your write-up. If it drops sharply on
`MarginRank` or `AchievementRank`, name the store that leads there instead, that is your
evidence that "highest sales" and "strongest performer" are not the same thing. Also check
whether the weakest store on `SalesRank` is weak everywhere, or whether it is actually
efficient (a strong `MarginRank`) despite low volume.

## Question 2 (4 marks) — Monthly Store Performance vs Target

**What I expect to find and why:** with 12 months and several stores, some store-months
will clear target and others won't. The interesting management question is whether misses
are scattered randomly, concentrated in specific months (pointing to a company-wide cause,
like a slow season), or concentrated in specific stores (pointing to a store-specific
issue, like understaffing or a badly set target).

**What the code does:** builds the Month × Store actual-vs-target table (as in Assignment
One, Question 9), then checks two things: whether achievement moves together across stores
in the same month (a company-wide pattern), and which single store-month is the most
unusual outlier.

In [ ]:
actual_month = master.groupby(["YearMonth", "StoreID", "StoreName"])["NetSales"] \
                      .sum().reset_index(name="ActualSales")

target_month = targets.groupby(["YearMonth", "StoreID"])["SalesTarget"].sum().reset_index()

combo = actual_month.merge(target_month, on=["YearMonth", "StoreID"], how="left")
combo["Variance"]        = combo["ActualSales"] - combo["SalesTarget"]
combo["AchievementPct"]  = combo["ActualSales"] / combo["SalesTarget"] * 100
combo["Status"]          = np.where(combo["Variance"] >= 0, "Met/Exceeded", "Missed")

print("Store-months meeting or exceeding target:", (combo["Status"] == "Met/Exceeded").sum(),
      "of", len(combo))

# Does achievement move together across stores in the same month? (company-wide pattern check)
pivot = combo.pivot(index="YearMonth", columns="StoreName", values="AchievementPct")
company_month_avg = pivot.mean(axis=1)
print("\nAverage achievement % by month, across all stores")
display(company_month_avg.to_frame("AvgAchievementPct").style.format("{:.1f}"))

# The single most unusual store-month (largest absolute variance)
extreme = combo.reindex(combo["Variance"].abs().sort_values(ascending=False).index).head(5)
print("MOST UNUSUAL STORE-MONTHS (largest variance, either direction)")
display(extreme[["YearMonth", "StoreName", "ActualSales", "SalesTarget", "Variance", "AchievementPct"]]
        .style.format({"ActualSales": "{:,.0f}", "SalesTarget": "{:,.0f}",
                        "Variance": "{:,.0f}", "AchievementPct": "{:.1f}"}))

# Visualization: heatmap of achievement % by store and month
heat = combo.pivot(index="StoreName", columns="YearMonth", values="AchievementPct")
plt.figure(figsize=(13, 5))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="RdYlGn", center=100,
            cbar_kws={"label": "Achievement %"})
plt.title("Target Achievement % by Store and Month (100 = on target)")
plt.tight_layout()
plt.show()

**How to interpret this:** look at `AvgAchievementPct` by month first, if it dips in the
same month across most stores, that points to a company-wide cause (a slow month, a
seasonal dip, all targets raised together that month) rather than a problem with one
store's management. Then look at the heatmap for a store whose row is a different colour
from every other store's row, in that month, that is the pattern or unusual result worth
naming. Whatever you pick, say clearly what management should check before acting on it,
for example whether the target itself was unrealistic, whether a promotion or stockout
happened that month, or whether the pattern repeats when more data becomes available.

# Case Study Two — Customer Value and Revenue Concentration (7 marks)

**The question management is really asking:** do all customers matter equally to the
business, or is value concentrated in a smaller group that deserves different treatment?

## Question 3 (4 marks) — Customer-Level and Customer-Type Analysis

**What I expect to find and why:** as with products and categories in Assignment One, I
expect the customer type with the most transactions to also have the highest total sales
(volume drives both), but its **share of profit** may not match its **share of sales**
exactly, because margin can differ by customer type (for example, if one type buys more
discounted or lower-margin products).

**What the code does:** builds a customer-level table (one row per CustomerID) and then
rolls it up to compare the three customer types on sales share vs profit share.

In [ ]:
customer_level = master.groupby(["CustomerID", "CustomerType"]).agg(
    TotalSales=("NetSales", "sum"),
    Transactions=("TransactionID", "nunique"),
    QuantityPurchased=("Quantity", "sum"),
    GrossProfit=("GrossProfit", "sum")
).reset_index()
customer_level["AvgTransactionValue"] = customer_level["TotalSales"] / customer_level["Transactions"]

print("CUSTOMER-LEVEL TABLE (sample)")
display(customer_level.sort_values("TotalSales", ascending=False).head(10)
        .style.format({"TotalSales": "{:,.0f}", "GrossProfit": "{:,.0f}",
                        "AvgTransactionValue": "{:,.0f}"}))

# Roll up to customer type
type_summary = customer_level.groupby("CustomerType").agg(
    Customers=("CustomerID", "nunique"),
    Transactions=("Transactions", "sum"),
    TotalSales=("TotalSales", "sum"),
    GrossProfit=("GrossProfit", "sum")
)
type_summary["SalesShare"]           = type_summary["TotalSales"] / type_summary["TotalSales"].sum()
type_summary["ProfitShare"]          = type_summary["GrossProfit"] / type_summary["GrossProfit"].sum()
type_summary["ProfitMargin"]         = type_summary["GrossProfit"] / type_summary["TotalSales"]
type_summary["AvgTransactionValue"]  = type_summary["TotalSales"] / type_summary["Transactions"]
type_summary["SalesPerCustomer"]     = type_summary["TotalSales"] / type_summary["Customers"]
type_summary = type_summary.sort_values("TotalSales", ascending=False)

print("CUSTOMER TYPE COMPARISON")
display(type_summary.style.format({
    "TotalSales": "{:,.0f}", "GrossProfit": "{:,.0f}", "SalesShare": "{:.2%}",
    "ProfitShare": "{:.2%}", "ProfitMargin": "{:.4f}",
    "AvgTransactionValue": "{:,.0f}", "SalesPerCustomer": "{:,.0f}"
}))

type_summary[["SalesShare", "ProfitShare"]].plot(kind="bar", figsize=(6, 4))
plt.title("Customer Type: Share of Sales vs Share of Profit")
plt.ylabel("Share of total")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**How to interpret this:** check whether the customer type with the highest `SalesShare`
also has the highest `ProfitShare`, and by how much its `ProfitMargin` differs from the
other types. Quote the actual percentages as your quantitative evidence. If the top-selling
type's profit share is clearly below its sales share, say so plainly, that is a real
finding, not a problem with the analysis. Also check `SalesPerCustomer`, since a type with
few customers but high sales per customer is a different story from a type with many
customers each buying a little.

## Question 4 (3 marks) — RFM Analysis

**What I expect to find and why:** RFM segments customers by **Recency** (how long since
their last purchase), **Frequency** (how many transactions they made) and **Monetary
value** (how much they spent). I expect most customers to cluster in the middle, with a
smaller group of clearly high-value, frequent, recent buyers, and another smaller group of
customers who haven't purchased in a long time, that second group is often where marketing
attention is most needed.

**What the code does:** calculates R, F and M per customer, scores each on a 1–4 scale
using quartiles, and combines the three scores into simple, named segments.

In [ ]:
snapshot_date = master["TransactionDate"].max() + pd.Timedelta(days=1)

rfm = master.groupby("CustomerID").agg(
    Recency=("TransactionDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("TransactionID", "nunique"),
    Monetary=("NetSales", "sum")
).reset_index()

# Score 1 (worst) to 4 (best) on each dimension using quartiles.
# Recency is reversed: a SMALLER number of days is better, so it gets labels in reverse order.
rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4]).astype(int)
rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

def segment(row):
    if row["R_Score"] >= 3 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
        return "Champions"
    if row["R_Score"] <= 2 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
        return "At Risk (were valuable)"
    if row["R_Score"] >= 3 and row["F_Score"] <= 2:
        return "New / Occasional"
    if row["R_Score"] <= 2 and row["F_Score"] <= 2 and row["M_Score"] <= 2:
        return "Lost / Low Value"
    return "Regular"

rfm["Segment"] = rfm.apply(segment, axis=1)

print("RFM TABLE (sample)")
display(rfm.head(10).style.format({"Monetary": "{:,.0f}"}))

seg_summary = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "nunique"),
    AvgRecency=("Recency", "mean"),
    AvgFrequency=("Frequency", "mean"),
    AvgMonetary=("Monetary", "mean"),
    TotalMonetary=("Monetary", "sum")
).sort_values("TotalMonetary", ascending=False)
seg_summary["ShareOfCustomers"] = seg_summary["Customers"] / seg_summary["Customers"].sum()
seg_summary["ShareOfRevenue"]   = seg_summary["TotalMonetary"] / seg_summary["TotalMonetary"].sum()

print("SEGMENT SUMMARY")
display(seg_summary.style.format({
    "AvgRecency": "{:.0f}", "AvgFrequency": "{:.1f}", "AvgMonetary": "{:,.0f}",
    "TotalMonetary": "{:,.0f}", "ShareOfCustomers": "{:.1%}", "ShareOfRevenue": "{:.1%}"
}))

plt.figure(figsize=(7, 4.5))
sns.scatterplot(data=rfm, x="Frequency", y="Monetary", hue="Segment", alpha=0.6)
plt.title("Customer Segments: Frequency vs Monetary Value")
plt.tight_layout()
plt.show()

**How to interpret this:** compare `ShareOfCustomers` with `ShareOfRevenue` for each
segment. A segment that is a small share of customers but a large share of revenue
(typically "Champions") is your evidence of concentrated value, similar in spirit to the
Pareto finding in Assignment One. Name the segment that needs attention (usually "At Risk"
or "Lost / Low Value", customers who used to buy often or spend a lot but haven't returned
recently) and explain in business terms what marketing might do differently for each group,
retain and reward Champions, win back At Risk customers, and encourage New/Occasional
customers to buy again.

# Case Study Three — Product Profitability and Discount Strategy (8 marks)

**The question management is really asking:** are the best-selling categories also the
most profitable ones, and are discounts actually helping or quietly hurting profit?

## Question 5 (4 marks) — Category Performance

**What I expect to find and why:** based on Assignment One's product-level findings,
different categories are likely to carry different margins, so I expect the top-selling
category by Net Sales to not necessarily be the top category by Gross Profit or margin.

**What the code does:** aggregates the master table by Category (Net Sales, Quantity,
Gross Profit, Profit Margin, Average Discount) and compares the sales ranking with the
profit ranking directly.

In [ ]:
category_summary = master.groupby("Category").agg(
    NetSales=("NetSales", "sum"),
    QuantitySold=("Quantity", "sum"),
    GrossProfit=("GrossProfit", "sum"),
    AvgDiscountPct=("DiscountPct", "mean")
).reset_index()
category_summary["ProfitMargin"] = category_summary["GrossProfit"] / category_summary["NetSales"]

category_summary["SalesRank"]  = category_summary["NetSales"].rank(ascending=False, method="min").astype(int)
category_summary["ProfitRank"] = category_summary["GrossProfit"].rank(ascending=False, method="min").astype(int)
category_summary = category_summary.sort_values("NetSales", ascending=False).reset_index(drop=True)

print("CATEGORY PERFORMANCE")
display(category_summary.style.format({
    "NetSales": "{:,.0f}", "GrossProfit": "{:,.0f}", "ProfitMargin": "{:.2%}",
    "AvgDiscountPct": "{:.3f}"
}))

plt.figure(figsize=(8, 5))
sns.scatterplot(data=category_summary, x="NetSales", y="ProfitMargin",
                size="GrossProfit", sizes=(200, 1200), hue="Category", legend=False)
for _, r in category_summary.iterrows():
    plt.annotate(r["Category"], (r["NetSales"], r["ProfitMargin"]),
                 xytext=(6, 6), textcoords="offset points", fontsize=8)
plt.title("Category Net Sales vs Profit Margin (bubble = Gross Profit)")
plt.tight_layout()
plt.show()

**How to interpret this:** compare `SalesRank` with `ProfitRank` for the top category. If
they match, say the top seller is genuinely the top earner and support that with the actual
Net Sales and Gross Profit figures. If they don't match, name which category is "high
sales, lower margin" and which is "lower sales, higher margin", and check `AvgDiscountPct`
too, a category with a much higher average discount is one place a low margin could be
coming from, which leads naturally into Question 6.

## Question 6 (4 marks) — Discount Percentage vs Profitability

**What I expect to find and why:** a bigger discount lowers the price without lowering the
cost, so I expect Gross Profit (and margin) to fall as `DiscountPct` rises. The real
question is whether that lost margin is being made up for by more units sold, if quantity
does not rise with discount, then discounting is not paying for itself.

**What the code does:** correlates `DiscountPct` with `GrossProfit` and `Quantity`, compares
average outcomes across discount levels, and visualizes the relationship.

In [ ]:
disc_cols = ["DiscountPct", "Quantity", "NetSales", "GrossProfit"]

print("CORRELATION WITH DISCOUNT PCT")
corr = master[disc_cols].corr(method="spearman")
display(corr)

by_discount = master.groupby("DiscountPct").agg(
    Transactions=("TransactionID", "count"),
    AvgQuantity=("Quantity", "mean"),
    AvgNetSales=("NetSales", "mean"),
    AvgGrossProfit=("GrossProfit", "mean"),
    TotalGrossProfit=("GrossProfit", "sum")
)
by_discount["GrossMargin"] = by_discount["TotalGrossProfit"] / (master.groupby("DiscountPct")["NetSales"].sum())

print("PERFORMANCE BY DISCOUNT LEVEL")
display(by_discount.style.format({
    "AvgQuantity": "{:.2f}", "AvgNetSales": "{:,.0f}", "AvgGrossProfit": "{:,.0f}",
    "TotalGrossProfit": "{:,.0f}", "GrossMargin": "{:.2%}"
}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
by_discount["AvgQuantity"].plot(kind="bar", ax=axes[0], title="Average Quantity by Discount")
by_discount["AvgGrossProfit"].plot(kind="bar", ax=axes[1], title="Average Gross Profit by Discount", color="tab:orange")
plt.tight_layout()
plt.show()

**How to interpret this:** state the Spearman correlation between `DiscountPct` and
`GrossProfit` (I expect this to be clearly negative) and between `DiscountPct` and
`Quantity` (I expect this to be close to zero). If `AvgQuantity` does not rise across
discount levels while `AvgGrossProfit` falls, that is your evidence that discounting is
reducing profit without a visible sales-volume payoff, an association, not proof that
discounts *cause* lower profit or that removing them would raise profit. **State one
limitation explicitly**, for example that discounts were not randomly assigned (they may be
concentrated on certain products or channels), so another factor could be driving the
pattern.

# Case Study Four — Sales Channel and Operational Performance (7 marks)

**The question management is really asking:** where should operational attention go,
which channels, customer groups or combinations need investigation?

## Question 7 (4 marks) — Sales Channel Comparison

**What I expect to find and why:** the channels (for example Store, Online, Corporate)
likely differ in typical basket size and margin, Corporate or bulk-style channels often
carry larger transaction values, while an online or retail channel may carry more frequent,
smaller transactions.

**What the code does:** aggregates by `SalesChannel` for Transactions, Net Sales, Gross
Profit, Profit Margin and Average Transaction Value.

In [ ]:
channel_summary = master.groupby("SalesChannel").agg(
    Transactions=("TransactionID", "nunique"),
    NetSales=("NetSales", "sum"),
    GrossProfit=("GrossProfit", "sum"),
    QuantitySold=("Quantity", "sum")
).reset_index()
channel_summary["ProfitMargin"]        = channel_summary["GrossProfit"] / channel_summary["NetSales"]
channel_summary["AvgTransactionValue"] = channel_summary["NetSales"] / channel_summary["Transactions"]
channel_summary["SalesShare"]          = channel_summary["NetSales"] / channel_summary["NetSales"].sum()
channel_summary = channel_summary.sort_values("NetSales", ascending=False)

print("SALES CHANNEL COMPARISON")
display(channel_summary.style.format({
    "NetSales": "{:,.0f}", "GrossProfit": "{:,.0f}", "ProfitMargin": "{:.2%}",
    "AvgTransactionValue": "{:,.0f}", "SalesShare": "{:.1%}"
}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=channel_summary, x="SalesChannel", y="AvgTransactionValue", ax=axes[0])
axes[0].set_title("Average Transaction Value by Channel")
sns.barplot(data=channel_summary, x="SalesChannel", y="ProfitMargin", ax=axes[1])
axes[1].set_title("Profit Margin by Channel")
plt.tight_layout()
plt.show()

**How to interpret this:** name the channel with the highest `AvgTransactionValue` and the
one with the highest `ProfitMargin`, if they're different channels, that's a useful finding
in itself. A channel with high sales share but low margin is a candidate for cost or pricing
review, while a small but high-margin channel could be worth expanding. Tie the finding back
to something operational, for example staffing an online channel differently from a
physical store channel.

## Question 8 (3 marks) — Customer Type × Sales Channel

**What I expect to find and why:** different customer types likely favour different
channels, for example Institutions or SMEs may buy more through a Corporate or Online
channel than through a physical Store, while Individuals may lean toward Store purchases.

**What the code does:** builds a cross-tabulation of CustomerType against SalesChannel
(row percentages, so each customer type sums to 100%) and visualizes it as a heatmap.

In [ ]:
cross = pd.crosstab(master["CustomerType"], master["SalesChannel"], normalize="index")
print("CUSTOMER TYPE x SALES CHANNEL (row %, each row sums to 100%)")
display(cross.style.format("{:.1%}"))

cross_counts = pd.crosstab(master["CustomerType"], master["SalesChannel"])
print("CUSTOMER TYPE x SALES CHANNEL (transaction counts)")
display(cross_counts)

plt.figure(figsize=(7, 4.5))
sns.heatmap(cross, annot=True, fmt=".1%", cmap="Blues")
plt.title("Share of Each Customer Type's Transactions, by Channel")
plt.tight_layout()
plt.show()

**How to interpret this:** look for a cell that stands out, either a customer type that is
unusually concentrated in one channel, or a channel that one customer type barely uses at
all. Name that specific combination as the operational issue worth investigating, and
explain why it matters, for example if Institutions rarely purchase online, that may point
to a missing B2B ordering feature or account-based sales process rather than a genuine lack
of demand. Support the claim with the actual percentage from the table, not just a general
impression.

# Final Management Question (Mandatory Integration Challenge)

**Task:** after completing all four case studies, identify the single most important
business insight in the dataset and answer three things: **what does the data show, what
does it mean for the business, and what should management investigate or do next.** This
must be backed by actual numbers from your analysis above, not a general statement.

**How to approach it:** don't introduce a new analysis here. Scan back through your outputs
from Questions 1–8 and pick the finding that is both **well supported by evidence** and
**most actionable** for the CEO, for example a store beating sales but missing target
consistently, a customer type or segment that drives disproportionate profit, a category
that is over-relied on for sales but weak on margin, or a discount pattern that is quietly
costing profit without lifting volume. The cell below pulls together a few of the headline
numbers from across the notebook so they're easy to compare side by side when you decide
which one to lead with.

In [ ]:
print("QUICK CROSS-CHECK OF HEADLINE FINDINGS (for choosing your key insight)")
print()
print("Store sales range: {:,.0f} to {:,.0f}".format(
    store_kpi['NetSales'].min(), store_kpi['NetSales'].max()))
print("Store target achievement range: {:.1f}% to {:.1f}%".format(
    store_kpi['AchievementPct'].min(), store_kpi['AchievementPct'].max()))
print()
print("Customer type sales share vs profit share:")
display(type_summary[["SalesShare", "ProfitShare"]].style.format("{:.1%}"))
print()
print("RFM segment share of customers vs share of revenue:")
display(seg_summary[["ShareOfCustomers", "ShareOfRevenue"]].style.format("{:.1%}"))
print()
print("Category sales share vs profit margin:")
display(category_summary.assign(SalesShare=lambda d: d['NetSales']/d['NetSales'].sum())
        [["Category", "SalesShare", "ProfitMargin"]].style.format({"SalesShare": "{:.1%}", "ProfitMargin": "{:.1%}"}))
print()
print("Discount level vs average gross profit:")
display(by_discount[["AvgGrossProfit", "GrossMargin"]].style.format({"AvgGrossProfit": "{:,.0f}", "GrossMargin": "{:.1%}"}))
print()
print("Sales channel share vs margin:")
display(channel_summary[["SalesChannel", "SalesShare", "ProfitMargin"]].style.format({"SalesShare": "{:.1%}", "ProfitMargin": "{:.1%}"}))

**Write your answer as three short paragraphs:**
1. **What the data shows** — state the specific numbers (percentages, amounts, ranks) behind
   your chosen insight.
2. **What it means for the business** — translate the numbers into plain business language,
   who is affected and how, in terms of revenue, profit or risk.
3. **What management should do next** — a concrete next step (investigate, test, reallocate,
   review), stated as something to look into or try, not a guaranteed fix, since this data
   shows association, not proof of cause.